In [ ]:
import pandas as pd 
#Imported all the Excel files 
df1 = pd.read_excel('filename.xlsx', engine='openpyxl')

# Display the first few rows_ I like to dsiplay the dataframe I am working with
print(df1.head())

#Merged the dataframe into one based on patient ID
df = pd.merge(df1, df2, on="PATIENT_ID")  #merge both dfs on the basis on ID 

#Dropped the column "LABORATORY_REFERENCE" since it is onlu lab-specific and does not relate to the study

df = df.drop(columns=["LABORATORY_REFERENCE"]) #Delete the column laboratory_reference 


#Replaced the words referring to sex with letters for better use of the memory
df["SEX"] = df["SEX"].replace({"FEMALE": "F", "MALE": "M"})

In [ ]:
#With less columns we can have the same amount of data. Created a column 'Age' with the info from date and birh and date of death. 

df['Age'] = None
#Dates must be converted into date objects
df["DATE_OF_BIRTH"] = pd.to_datetime(df["DATE_OF_BIRTH"], dayfirst=True, errors="coerce")
df["DATE_OF_DEATH"] = pd.to_datetime(df["DATE_OF_DEATH"], dayfirst=True, errors="coerce")

# Compute age in years (difference)
#If there is no data of death the age is calculated from date of birth untill the day of data handeling (logging date is unavailable)
df["Age"] = ((df["DATE_OF_DEATH"].fillna(pd.Timestamp.today()) - df["DATE_OF_BIRTH"]).dt.days / 365.25).astype(int)


print("\nUpdated DataFrame with age:")

df = df.drop(columns=["DATE_OF_BIRTH"]) 
df = df.drop(columns=["DATE_OF_DEATH"]) 
print(df)

In [ ]:
#Counted the patients who are under 60 years of age
count_under_60 = (df["Age"] < 60).sum()
print(count_under_60)

#deleted the patients who are under 60 years of age. The study focuses on this age 
df_over60 = df[df["Age"] >= 60]
print(df_over60.shape)

In [ ]:
#import the diagnoses columns and minimize the working space to include study-specific terms 
dg = pd.read_excel('Diagnosis_transposed.xlsx', engine='openpyxl')

search_terms = ["nerv", "cogniti", "brain", "memory", "alzheimer", "dementia", "diab"]

# Find all columns that contain ANY of the search terms (case-insensitive)
cols1 = [col for col in dg.columns if any(term in col.lower() for term in search_terms)]

print("Matching columns:", cols1)

In [ ]:
df_col = pd.DataFrame(cols1, columns=["values"])
print(df_col)
df_col_T = df_col.T
df_col_T.columns = df_col_T.iloc[0]      # set first row as header
df_col_T = df_col_T.drop(df_col_T.index[0])    # drop the first row
df_col_T = df_col_T.reset_index(drop=True)  # reset index
df_col_T = df_col_T[sorted(df_col_T.columns)]
print(df_col_T.shape)


In [ ]:
#Combined the main dataframe with the transposed diagnosis ( each diagnosis in a column)
df_combined = pd.concat([df_over60, df_col_T], axis=0)

In [ ]:
# Surf through rows in DIAGNOSIS and mark 1 if the diagnosis contains the column name
for col in df_combined.columns:
    if col != "DIAGNOSIS":  # skip the DIAGNOSIS column itself
        df_combined[col] = df_combined.apply(
            lambda row: 1 if str(col).lower() in str(row["DIAGNOSIS"]).lower() else 0,
            axis=1
        )

print(df_combined)

In [ ]:
#Have it in an Excel sheet as required by the analysis scientist
df_combined.to_excel('path/final.xlsx', sheet_name='final', index=False)